In [ ]:
import argparse
import os
import mmcv
from mmdet.apis import init_detector, inference_detector
from mmdet.registry import VISUALIZERS

In [ ]:
def parse_args():
    parser = argparse.ArgumentParser(description='MMDetection inference')
    parser.add_argument(
        '--config',
        required=True,
        help='Path to model config file'
    )
    parser.add_argument(
        '--checkpoint',
        required=True,
        help='Path to trained checkpoint'
    )
    parser.add_argument(
        '--img-dir',
        required=True,
        help='Directory containing input images'
    )
    parser.add_argument(
        '--out-dir',
        default='inference_results',
        help='Directory to save visualized results'
    )
    parser.add_argument(
        '--score-thr',
        type=float,
        default=0.5,
        help='Score threshold'
    )
    parser.add_argument(
        '--device',
        default='cuda:0',
        help='cuda:0 or cpu'
    )
    return parser.parse_args()

In [ ]:
def inference():
    args = parse_args()

    os.makedirs(args.out_dir, exist_ok=True)

    # ------------------------------------------------
    # Load model
    # ------------------------------------------------
    print(f'Loading model:\n  config: {args.config}\n  ckpt: {args.checkpoint}')
    model = init_detector(args.config, args.checkpoint, device=args.device)

    # ------------------------------------------------
    # Visualizer
    # ------------------------------------------------
    visualizer = VISUALIZERS.build(model.cfg.visualizer)
    visualizer.dataset_meta = model.dataset_meta

    # ------------------------------------------------
    # Collect images
    # ------------------------------------------------
    img_files = [
        f for f in os.listdir(args.img_dir)
        if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))
    ]

    print(f'Found {len(img_files)} images')
    print('-' * 50)

    # ------------------------------------------------
    # Inference loop
    # ------------------------------------------------
    for img_name in img_files:
        img_path = os.path.join(args.img_dir, img_name)
        print(f'Processing: {img_name}')

        # Run inference
        result = inference_detector(model, img_path)

        # Read image
        img = mmcv.imread(img_path, channel_order='rgb')

        # Draw prediction
        visualizer.add_datasample(
            name=img_name,
            image=img,
            data_sample=result,
            draw_gt=False,
            pred_score_thr=args.score_thr,
            show=False,
            out_file=os.path.join(args.out_dir, img_name)
        )

        # Optional: print bbox info
        pred = result.pred_instances
        keep = pred.scores >= args.score_thr
        print(f'  Detections: {keep.sum().item()}')

    print('\nInference finished.')
    print(f'Results saved to: {args.out_dir}')

In [ ]:
inference()